In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import os

# Load environment variables from .env file
load_dotenv()


## Creating tools

In [2]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  # Get API key from environment variables
  api_key = os.getenv('EXCHANGE_RATE_API_KEY', '')
  
  # If no API key is found, provide a helpful message
  if not api_key:
    return {
      "error": "API key not found. Please set EXCHANGE_RATE_API_KEY in your .env file.",
      "instructions": "Get your API key from https://www.exchangerate-api.com/"
    }
    
  url = f'https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}'
  
  response = requests.get(url)
  
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [3]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

## Tool Binding

In [4]:
llm = ChatGroq(
    model_name="deepseek-r1-distill-llama-70b",
)

In [5]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [6]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [7]:
import json

# Example of how to use the tools
def process_currency_conversion(base_currency, target_currency, amount):
    # Get conversion factor
    conversion_result = get_conversion_factor.invoke({
        'base_currency': base_currency,
        'target_currency': target_currency
    })
    
    # Check if there was an error
    if 'error' in conversion_result:
        return conversion_result
    
    # Extract conversion rate
    conversion_rate = conversion_result['conversion_rate']
    
    # Convert the amount
    converted_amount = convert.invoke({
        'base_currency_value': amount,
        'conversion_rate': conversion_rate
    })
    
    return {
        'base_currency': base_currency,
        'target_currency': target_currency,
        'amount': amount,
        'conversion_rate': conversion_rate,
        'converted_amount': converted_amount
    }


## Example Usage

To use this tool, you need to:

1. Create a `.env` file with your API keys:
```
GROQ_API_KEY=your_groq_api_key_here
EXCHANGE_RATE_API_KEY=your_exchange_rate_api_key_here
```

2. Run the notebook cells above

3. Use the LLM to process natural language requests for currency conversion:
```python
response = llm_with_tools.invoke(messages)
print(response.content)
```

4. Or use the tools directly:
```python
result = process_currency_conversion('USD', 'EUR', 100)
print(f"100 USD = {result['converted_amount']} EUR")
```